#### Data

In [ ]:
%load_ext autoreload
%autoreload 2
import random
import numpy as np
from typing import List
import math
import backups.vehicle_routing_problem as gavrp

In [ ]:
NODES = [1,2,3,4,5,6,7,8,9]
# initial_route = [3,6,5,4,2,8,9,7,1]
TIME_MATRIX = {(i,j):np.random.randint(5,10) for i in NODES + [0] for j in NODES + [0]}
DEMAND = {i:1 for i in NODES}
VEHICLES = [1,2,3]
CAPACITY = {i:4 for i in VEHICLES}
TIME_WINDOW = {
    0: {'start_time': 0, 'end_time': 10},
    1: {'start_time': 45, 'end_time': 146},
    2: {'start_time': 50, 'end_time': 155},
    3: {'start_time': 80, 'end_time': 150},
    4: {'start_time': 45, 'end_time': 153},
    5: {'start_time': 35, 'end_time': 184},
    6: {'start_time': 78, 'end_time': 176},
    7: {'start_time': 73, 'end_time': 160},
    8: {'start_time': 60, 'end_time': 144},
    9: {'start_time': 79, 'end_time': 150},
    # 10: {'start_time': 35, 'end_time': 184},
    # 11: {'start_time': 78, 'end_time': 176},
    # 12: {'start_time': 73, 'end_time': 160},
    # 13: {'start_time': 60, 'end_time': 144},
    # 14: {'start_time': 79, 'end_time': 150}
}

In [ ]:
subroutes = gavrp.route_to_subroute(
    route = gavrp.generate_random_route(nodes = NODES),
    vehicles = VEHICLES,
    time_matrix= TIME_MATRIX,
    demand_by_node = DEMAND,
    capacity_by_vehicle = CAPACITY,
    flexibility_factor = 0.15,
    time_window_by_node=TIME_WINDOW
)
# gavrp.is_feasible(subroute_by_vehicle=subroutes,n_nodes=len(NODES))

In [ ]:
gavrp.initialize_population(
    route = gavrp.generate_random_route(nodes = NODES),
    vehicles = VEHICLES,
    time_matrix= TIME_MATRIX,
    demand_by_node = DEMAND,
    capacity_by_vehicle = CAPACITY,
    flexibility_factor = 0.15,
    time_window_by_node=TIME_WINDOW
)

In [ ]:
def fitness(subroute_by_vehicle,time_matrix = TIME_MATRIX):
    total_time = 0
    for vehicle in subroute_by_vehicle:
        route = [0] + subroute_by_vehicle[vehicle] + [0]
        for i in range(len(route)-1):
            total_time = total_time + time_matrix[route[i],route[i+1]]
    return 1/total_time

def selection(population, fitness_func):
    return random.choices(
        population=population,
        weights=[fitness_func(gene) for gene in population],
        k=2
    )

In [ ]:
fitness_score = 10*10
fitness_score_new = fitness(subroute_by_vehicle=subroutes)
population = gavrp.initialize_population(
    route = gavrp.generate_random_route(nodes = NODES),
    vehicles = VEHICLES,
    time_matrix= TIME_MATRIX,
    demand_by_node = DEMAND,
    capacity_by_vehicle = CAPACITY,
    flexibility_factor = 0.15,
    time_window_by_node=TIME_WINDOW
)
fitness_limit = 10*10
obj_list = []
for i in range(1000):
    population = sorted(population, key=lambda genome: fitness(genome), reverse=True)
    obj_list.append(1/fitness(population[0]))

    if fitness(population[0]) >= fitness_limit:
        break
    next_generation = population[0:2]
    for j in range(int(len(population) / 2) - 1):
        # Selection
        parents = selection(population, fitness)
        offspring_a, offspring_b = gavrp.crossover(
            subroute_a = parents[0], 
            subroute_b = parents[1],
            vehicles = VEHICLES,
            time_matrix= TIME_MATRIX,
            demand_by_node = DEMAND,
            capacity_by_vehicle = CAPACITY,
            flexibility_factor = 0.15,
            time_window_by_node=TIME_WINDOW
        )
        # Mutation
        offspring_a = gavrp.mutation(
            subroute = offspring_a,
            vehicles = VEHICLES,
            time_matrix= TIME_MATRIX,
            demand_by_node = DEMAND,
            capacity_by_vehicle = CAPACITY,
            flexibility_factor = 0.15,
            time_window_by_node=TIME_WINDOW
        )
        # Crossover
        offspring_b = gavrp.mutation(
            subroute = offspring_b,
            vehicles = VEHICLES,
            time_matrix= TIME_MATRIX,
            demand_by_node = DEMAND,
            capacity_by_vehicle = CAPACITY,
            flexibility_factor = 0.15,
            time_window_by_node=TIME_WINDOW
        )
        # Next generation
        next_generation += [offspring_a, offspring_b]
    population = next_generation
population = sorted(population, key=lambda genome: fitness(genome), reverse=True)
print(population[0])
print(1/fitness(population[0]))

#### genetic Algorithm

In [1]:
%load_ext autoreload
%autoreload 2
import polars as pl
from metaheuristics.cvrptw import *
from metaheuristics.genetic_algorithm import *

In [2]:
data = pl.read_csv("data/data.csv")
data = data.filter(pl.col("node_type") != "depot")
data = data.with_columns(
    pl.col("start_time")
    .str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S')
    .alias("start_time")
)
data = data.with_columns(
    pl.col("end_time")
    .str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S')
    .alias("end_time")
)
data.head()

X,Y,node_type,node_nbr,start_time,end_time
f64,f64,str,i64,datetime[μs],datetime[μs]
0.47,1.48,"""non-depot""",1,2024-07-07 00:00:00,2024-07-07 23:59:59
4.86,-4.66,"""non-depot""",2,2024-07-07 00:00:00,2024-07-07 23:59:59
-3.93,-2.57,"""non-depot""",3,2024-07-07 00:00:00,2024-07-07 23:59:59
2.14,2.89,"""non-depot""",4,2024-07-07 00:00:00,2024-07-07 23:59:59
-3.04,-0.14,"""non-depot""",5,2024-07-07 00:00:00,2024-07-07 23:59:59


In [3]:
nodes = []
for row in data.rows(named = True):
    nodes.append(
        Node(
            id = str(row["node_nbr"]),
            longitude = row["X"],
            latitude = row["Y"],
            demand = 1,
            ready_timestamp=row["start_time"],
            due_timestamp = row["end_time"],
            service_time_sec=0
        )
    )
depot_node = Node(
    id = "0",
    longitude=0,
    latitude = 0,
    demand = 0,
    ready_timestamp=data["start_time"].min(),
    due_timestamp=data["end_time"].max(),
    service_time_sec=0
)
N_VEHICLE = 23
vehicles = []
for i in range(N_VEHICLE):
    vehicles.append(
        Vehicle(
            id = str(i),
            capacity = 5,
            speed_per_sec=1,
            depot_node=depot_node
        )
    )

In [4]:
utils = CVRPTWGeneticAlgorithmUtils(
    nodes = nodes,
    vehicles=vehicles,
    population_size=100
)

In [5]:
ga = GeneticAlgorithm(
    iteration = 100,
    fitness_limit = 10*10,
    population_init_function = utils.generate_population,
    selection_function = utils.selection_function,
    fitness_function = utils.calculate_fitness,
    crossover_function = utils.crossover,
    mutation_function = utils.mutate,
)

In [6]:
ga = ga.optimize()
optimal_solution = ga.find_best_solution()

The best objective is: 0.0019245971883872386 & best solution: CVRPTWGenome(vehicles=[Vehicle(id='0', capacity=5.0, speed_per_sec=1.0, depot_node=Node(id='0', longitude=0.0, latitude=0.0, demand=0.0, ready_timestamp=datetime.datetime(2024, 7, 7, 0, 0), due_timestamp=datetime.datetime(2024, 7, 7, 23, 59, 59), service_time_sec=0.0)), Vehicle(id='1', capacity=5.0, speed_per_sec=1.0, depot_node=Node(id='0', longitude=0.0, latitude=0.0, demand=0.0, ready_timestamp=datetime.datetime(2024, 7, 7, 0, 0), due_timestamp=datetime.datetime(2024, 7, 7, 23, 59, 59), service_time_sec=0.0)), Vehicle(id='2', capacity=5.0, speed_per_sec=1.0, depot_node=Node(id='0', longitude=0.0, latitude=0.0, demand=0.0, ready_timestamp=datetime.datetime(2024, 7, 7, 0, 0), due_timestamp=datetime.datetime(2024, 7, 7, 23, 59, 59), service_time_sec=0.0)), Vehicle(id='3', capacity=5.0, speed_per_sec=1.0, depot_node=Node(id='0', longitude=0.0, latitude=0.0, demand=0.0, ready_timestamp=datetime.datetime(2024, 7, 7, 0, 0), due_

In [7]:
1/optimal_solution.best_fitness_score

519.5892449775288